In [0]:
%python

''' 1. IMPORTS Y SETUPS '''

import pyspark.sql 
from pyspark.errors import PySparkException
import json
from pyspark.sql.functions import col, current_timestamp, max as spark_max
from delta.tables import DeltaTable #Data Lake adds ACID transactions, UPDATE/DELETE/MERGE support, and time travel to Spark DataFrames stored as Parquet

'''
 2. JSON METADATA LOAD  -> Dict
'''
json_metadata = [ 
                 #Relational Tables / Bank Core
                {  "table_name": "customers",
                    "source_table": "banking.customers", 
                    "target_table": "banking_bronze.customers"
                },
                {  "table_name": "branches",
                    "source_table": "banking.branches", 
                    "target_table": "banking_bronze.branches"
                }, 
                {   "table_name": "accounts",
                    "source_table": "banking.accounts", 
                    "target_table": "banking_bronze.accounts"},
                {   "table_name": "transactions", 
                    "source_table": "banking.transactions", 
                    "target_table": "banking_bronze.transactions"},
    
                # Non relational tables/ external 
                {   "table_name": "credit_bureau_reports", 
                    "source_table": "default.credit_bureau_reports", 
                    "target_table": "banking_bronze.credit_bureau_reports"},
                {   "table_name": "payment_gateway_logs",   
                    "source_table": "default.payment_gateway_logs", 
                    "target_table": "banking_bronze.payment_gateway_logs"}
                ]


''' 
3 TRY-CATCH AUDIT CONTROL
'''
    # Adding data of tables into BRONZE table
try: 
    for metad in json_metadata:

        #df = spark.table("scheme_name.table_name")
        df = spark.table(metad["source_table"])
        df.write.format("delta")\
            .mode("overwrite")\
            .saveAsTable(metad["target_table"])

    print("Ingest data success!")
except KeyError as es: 
    print(f"METADATA ERROR: Key {es} is missing from the dictionary")
except NameError as namerr:
    print(f"NON-EXISTENT DICTIONARY: Dictionary variable IS NOT {namerr} or does not exist.")
except PySparkException as pyErr: 
    print(f"PySpark Error: Missing permissions, table mismatch, non-existent tables, etc. -- \n {pyErr}")
except Exception as ex: 
    print(f"Other Error: {ex}")
finally:
    print("End of bronze stage")


